 ## Training ML on Large Encrypted Datasets with Blind Insight
 ### **Six features, one target variable, three models**

 **Scenario:**
  Models get smarter with combined datasets. Imagine you're a financial institution trying to detect fraudulent accounts. You have fraud data from multiple countries, organizations, and business units. Traditional ML requires decrypted, plaintext data.

 **Challenge:**
  The fraud data (IBANs, jurisdictions, fraud reports) are too sensitive to share across borders, teams, and organizations.
  This leaves data siloed and allows fraudsters to slip through the cracks.

 **Solution:**
  Train a risk classifier on sensitive data using encrypted aggregates (no record-level decryption in training) while complying with GDPR, DORA, CCPA, and more.

 ### Import Dependencies

In [1]:
import warnings; warnings.filterwarnings('ignore')
import os, time
from IPython.display import display, HTML
from blind_ml.client import BlindInsightClient
from blind_ml.demo_helpers import (
    load_env, get_fraud_demo_config, load_training_data, load_test_data,
    discover_feature_values, run_bi_training,
    train_plaintext_nb, naive_bayes_predict,
)

# --- Configuration ---
cfg = get_fraud_demo_config()
load_env()  # BI_EMAIL, BI_PASSWORD, BI_ORG from .env
PROXY_URL = os.environ.get("BI_PROXY_URL", "https://local.blindinsight.io")
ORG = os.environ.get("BI_ORG", "your-org-slug")
DATASET = cfg["dataset"]
SCHEMA = cfg["schema"]
SQLITE_DB = cfg["sqlite_db"]
TEST_SQLITE_DB = cfg["test_sqlite_db"]
FEATURES = cfg["features"]
TARGET = cfg["target"]

# --- Connect to Blind Insight proxy (handles encryption/decryption) ---
client = BlindInsightClient(proxy_url=PROXY_URL, verify_ssl=False)
client.warm_up(ORG, DATASET, SCHEMA)

# --- Load plaintext training data (local mirror of encrypted BI data) ---
df, TRAIN_LIMIT = load_training_data(SQLITE_DB)
_, TEST_LIMIT   = load_test_data(TEST_SQLITE_DB)
print(f"Proxy: {PROXY_URL} | Train: {TRAIN_LIMIT:,} records | Test: {TEST_LIMIT:,} records")

  Proxy warm-up: schema (3350ms) + 7 index preflights (2757ms) = 6107ms total
Proxy: https://local.blindinsight.io | Train: 500,000 records | Test: 50,000 records


 ### Training Set Sample Records

In [2]:
from blind_ml.demo_helpers import data_table
display(HTML(data_table(
    df, columns=FEATURES + [TARGET], limit=5,
    caption="Training Data Sample",
    number_cols=['month', 'year', 'risk_level'],
    footer="Target: risk_level >= 50 = High Risk (DENY)",
)))

fraud_type,account_jurisdiction,is_active,month,reporting_bank_id,year,risk_level
account_takeover,DE,true,7,BANK007,2018,51
identity_theft,US,false,5,BANK027,2021,82
unusual_activity,DE,false,2,BANK022,2018,10
account_takeover,US,true,10,BANK036,2024,52
mule_account,AU,false,8,BANK015,2021,54


 ### Train Naive Bayes on Encrypted Data

 Train the same model on plaintext for benchmarking comparison.

In [3]:
from blind_ml.demo_helpers import training_summary_table

print("Training models...")
feature_values = discover_feature_values(df)
feature_values["bank_ids"] = ["BANK001", "BANK002", "BANK003", "BANK014"]

n_high_local = int((df["risk_level"] >= 50).sum())
n_low_local  = len(df) - n_high_local

# --- Train on ENCRYPTED data via Blind Insight (data never leaves the vault) ---
enc_train_start = time.time()
print("  Encrypted (Blind Insight)...", end=" ", flush=True)
client.profiling.enable()
bi = run_bi_training(
    client, ORG, DATASET, SCHEMA, feature_values,
    n_high_local=n_high_local, n_low_local=n_low_local,
)
client.profiling.disable()
enc_train_time = time.time() - enc_train_start
print(f"done ({enc_train_time:.1f}s)")

# --- Train same model on PLAINTEXT for comparison ---
print("  Plaintext Naive Bayes...", end=" ", flush=True)
plain_nb = train_plaintext_nb(df, feature_values)
print("done")

# Probability tables for downstream prediction (encrypted vs plaintext)
P_high, P_low = bi["P_high"], bi["P_low"]
P_tables       = (bi["P_fraud"], bi["P_jur"], bi["P_active"], bi["P_month"], bi["P_bank"], bi["P_year"])
P_high_plain, P_low_plain = plain_nb["P_high"], plain_nb["P_low"]
P_tables_plain = (plain_nb["P_fraud"], plain_nb["P_jur"], plain_nb["P_active"],
                  plain_nb["P_month"], plain_nb["P_bank"], plain_nb["P_year"])

enc_queries    = bi["enc_queries"]
plain_train_time = len(df) * 1e-6

# --- Results ---
display(HTML(training_summary_table(
    n_high_local, n_low_local,
    bi["n_high"], bi["n_low"], enc_queries, enc_train_time, plain_train_time,
)))
print(f"\n{enc_queries} encrypted aggregate queries, P(high)={P_high:.3f}")

Training models...
  Encrypted (Blind Insight)...   Base rates (BI):    1,320 records, high=840, low=480
  Base rates (local sanity): 500,000 records, high=326,472, low=173,528
done (14.6s)
  Plaintext Naive Bayes... done


,Plaintext,Blind Insight,Overhead
High Risk,"326,472",840,-
Low Risk,"173,528",480,-
Total,"500,000","1,320",-
Queries,0,90,-
Train Time,0.500000s,14.6s,+14.1s
Data Decrypted,YES,NEVER,-



90 encrypted aggregate queries, P(high)=0.636


 ### Train Gaussian Naive Bayes on Encrypted Data

 Build a Gaussian Naive Bayes model from encrypted count summaries over numeric
 fraud fields (`month`, `day`, `year`). Compare against sklearn's
 `GaussianNB` trained on the plaintext local mirror.


In [4]:
from blind_ml.demo_helpers import (
    run_encrypted_gnb_fraud, fraud_gnb_predict,
    train_plaintext_gnb_fraud, fraud_plaintext_gnb_predict_proba,
    fraud_model_summary_table, fraud_confusion_matrix_html,
    training_summary_table, load_test_data, discover_feature_values, get_bi_base_rates,
    compute_fraud_metrics,
)

print("Training Gaussian Naive Bayes...")

if "feature_values" not in globals():
    feature_values = discover_feature_values(df)
    feature_values["bank_ids"] = ["BANK001", "BANK002", "BANK003", "BANK014"]
if "day_values" not in feature_values:
    feature_values["day_values"] = sorted(df["day"].astype(str).unique().tolist(), key=lambda x: int(x))

GNB_FEATURES = ["month", "day", "year"]
n_high_gnb, n_low_gnb = get_bi_base_rates(client, ORG, DATASET, SCHEMA)

# --- Encrypted GaussianNB ---
print("  Encrypted GaussianNB...", end=" ", flush=True)
gnb_enc = run_encrypted_gnb_fraud(
    client, ORG, DATASET, SCHEMA, feature_values,
    numeric_features=GNB_FEATURES,
    n_high=n_high_gnb, n_low=n_low_gnb,
)
print(f"done ({gnb_enc['train_time']:.1f}s)")

# --- sklearn GaussianNB benchmark ---
print("  sklearn GaussianNB...", end=" ", flush=True)
gnb_plain = train_plaintext_gnb_fraud(df, numeric_features=GNB_FEATURES)
print(f"done ({gnb_plain['train_time']*1000:.0f}ms)")

# --- Results ---
display(HTML(training_summary_table(
    gnb_plain["n_high"], gnb_plain["n_low"],
    gnb_enc["n_high"], gnb_enc["n_low"],
    gnb_enc["enc_queries"], gnb_enc["train_time"], gnb_plain["train_time"],
)))
print(f"\n{gnb_enc['enc_queries']} encrypted aggregate queries, P(high)={gnb_enc['_model'].P_pos:.3f}")

# --- Evaluate on test set ---
if "df_test_dt" not in globals():
    df_test_dt, _ = load_test_data(TEST_SQLITE_DB)
if "COHORT_PRIOR" not in globals():
    COHORT_PRIOR = float(df_test_dt["is_high_risk"].mean())

y_true_gnb = df_test_dt["is_high_risk"].values

gnb_enc_scores = []
for _, row in df_test_dt.iterrows():
    _, risk = fraud_gnb_predict(gnb_enc, row.to_dict())
    gnb_enc_scores.append(risk)
gnb_enc_m = compute_fraud_metrics(y_true_gnb, gnb_enc_scores, cohort_prior=COHORT_PRIOR)

gnb_plain_proba = fraud_plaintext_gnb_predict_proba(
    gnb_plain["model"], gnb_plain["features"], df_test_dt,
)
gnb_plain_m = compute_fraud_metrics(y_true_gnb, gnb_plain_proba, cohort_prior=COHORT_PRIOR)

print(
    f"\nEncrypted GaussianNB F1={gnb_enc_m['f1']:.3f} ROC-AUC={gnb_enc_m['roc_auc']:.3f} | "
    f"sklearn F1={gnb_plain_m['f1']:.3f} ROC-AUC={gnb_plain_m['roc_auc']:.3f}"
)
display(HTML(fraud_model_summary_table(
    "Gaussian Naive Bayes",
    enc_metrics=gnb_enc_m,
    plain_metrics=gnb_plain_m,
    enc_train_time=gnb_enc["train_time"],
    plain_train_time=gnb_plain["train_time"],
    enc_queries=gnb_enc["enc_queries"],
    plain_label="sklearn GaussianNB",
)))
display(HTML(fraud_confusion_matrix_html("Gaussian Naive Bayes", gnb_enc_m, gnb_plain_m)))


Training Gaussian Naive Bayes...
  Encrypted GaussianNB... done (10.8s)
  sklearn GaussianNB... done (203ms)


,Plaintext,Blind Insight,Overhead
High Risk,"326,472",840,-
Low Risk,"173,528",480,-
Total,"500,000","1,320",-
Queries,0,96,-
Train Time,0.203244s,10.8s,+10.6s
Data Decrypted,YES,NEVER,-



96 encrypted aggregate queries, P(high)=0.636

Encrypted GaussianNB F1=0.789 ROC-AUC=0.505 | sklearn F1=0.789 ROC-AUC=0.493


,sklearn GaussianNB,Blind Insight,Delta
F1 @0.5 (demo prior),0.789,0.789,+0.000
F1@best (demo prior),0.789,0.789,+0.000
ROC-AUC,0.493,0.505,+0.012
PR-AUC,0.648,0.654,+0.006
F1@best @ 1.5% prod prior,0.789,0.789,+0.000
Sensitivity @0.5,100.0%,100.0%,+0.000
Specificity @0.5,0.0%,0.0%,+0.000
PPV (precision) @0.5,65.1%,65.1%,+0.000
Flagged High-Risk @0.5,100.0%,100.0%,+0.000
Train Time,203ms,10.8s,+10.6s


,Pred Low,Pred High
Actual Low,0,"17,455"
Actual High,0,"32,545"
,Pred Low,Pred High
Actual Low,0,"17,455"
Actual High,0,"32,545"


 ### Train Bayesian Network on Encrypted Data


In [5]:
from blind_ml.demo_helpers import (
    run_encrypted_bn_fraud, fraud_bn_predict,
    train_plaintext_bn_fraud, fraud_plaintext_bn_predict_proba,
    fraud_model_summary_table, fraud_confusion_matrix_html,
    training_summary_table, load_test_data, discover_feature_values, get_bi_base_rates,
    compute_fraud_metrics,
)

print("Training Bayesian Network...")

if "feature_values" not in globals():
    feature_values = discover_feature_values(df)
    feature_values["bank_ids"] = ["BANK001", "BANK002", "BANK003", "BANK014"]

BN_PARENT_MAP = {
    "fraud_type": [],
    "account_jurisdiction": ["fraud_type"],
    "is_active": ["fraud_type"],
    "month": ["year"],
    "reporting_bank_id": ["account_jurisdiction"],
    "year": [],
}

n_high_bn, n_low_bn = get_bi_base_rates(client, ORG, DATASET, SCHEMA)

# --- Encrypted Bayesian Network ---
print("  Encrypted Bayesian Network CPTs...", end=" ", flush=True)
bn_enc = run_encrypted_bn_fraud(
    client, ORG, DATASET, SCHEMA, feature_values,
    parent_map=BN_PARENT_MAP,
    n_high=n_high_bn, n_low=n_low_bn,
)
print(f"done ({bn_enc['train_time']:.1f}s)")

# --- Plaintext Bayesian Network benchmark ---
print("  Plaintext Bayesian Network...", end=" ", flush=True)
bn_plain = train_plaintext_bn_fraud(
    df, feature_values,
    parent_map=BN_PARENT_MAP,
)
print(f"done ({bn_plain['train_time']*1000:.0f}ms)")

# --- Results ---
display(HTML(training_summary_table(
    bn_plain["n_high"], bn_plain["n_low"],
    bn_enc["n_high"], bn_enc["n_low"],
    bn_enc["enc_queries"], bn_enc["train_time"], bn_plain["train_time"],
)))
print(f"\n{bn_enc['enc_queries']} encrypted CPT queries, P(high)={bn_enc['_model'].P_pos:.3f}")

# --- Evaluate on test set ---
if "df_test_dt" not in globals():
    df_test_dt, _ = load_test_data(TEST_SQLITE_DB)
if "COHORT_PRIOR" not in globals():
    COHORT_PRIOR = float(df_test_dt["is_high_risk"].mean())

y_true_bn = df_test_dt["is_high_risk"].values

bn_enc_scores = []
for _, row in df_test_dt.iterrows():
    _, risk = fraud_bn_predict(bn_enc, row.to_dict())
    bn_enc_scores.append(risk)
bn_enc_m = compute_fraud_metrics(y_true_bn, bn_enc_scores, cohort_prior=COHORT_PRIOR)

bn_plain_proba = fraud_plaintext_bn_predict_proba(bn_plain, df_test_dt)
bn_plain_m = compute_fraud_metrics(y_true_bn, bn_plain_proba, cohort_prior=COHORT_PRIOR)

print(
    f"\nEncrypted BN F1={bn_enc_m['f1']:.3f} ROC-AUC={bn_enc_m['roc_auc']:.3f} | "
    f"Plaintext BN F1={bn_plain_m['f1']:.3f} ROC-AUC={bn_plain_m['roc_auc']:.3f}"
)
display(HTML(fraud_model_summary_table(
    "Bayesian Network",
    enc_metrics=bn_enc_m,
    plain_metrics=bn_plain_m,
    enc_train_time=bn_enc["train_time"],
    plain_train_time=bn_plain["train_time"],
    enc_queries=bn_enc["enc_queries"],
    plain_label="Plaintext BN",
)))
display(HTML(fraud_confusion_matrix_html("Bayesian Network", bn_enc_m, bn_plain_m)))


Training Bayesian Network...
  Encrypted Bayesian Network CPTs... done (54.9s)
  Plaintext Bayesian Network... done (1230ms)


,Plaintext,Blind Insight,Overhead
High Risk,"326,472",840,-
Low Risk,"173,528",480,-
Total,"500,000","1,320",-
Queries,0,514,-
Train Time,1.230430s,54.9s,+53.7s
Data Decrypted,YES,NEVER,-



514 encrypted CPT queries, P(high)=0.636

Encrypted BN F1=1.000 ROC-AUC=1.000 | Plaintext BN F1=1.000 ROC-AUC=1.000


,Plaintext BN,Blind Insight,Delta
F1 @0.5 (demo prior),1.000,1.000,+0.000
F1@best (demo prior),1.000,1.000,+0.000
ROC-AUC,1.000,1.000,+0.000
PR-AUC,1.000,1.000,+0.000
F1@best @ 1.5% prod prior,1.000,1.000,+0.000
Sensitivity @0.5,100.0%,100.0%,+0.000
Specificity @0.5,100.0%,100.0%,+0.000
PPV (precision) @0.5,100.0%,100.0%,+0.000
Flagged High-Risk @0.5,65.1%,65.1%,+0.000
Train Time,1230ms,54.9s,+53.7s


,Pred Low,Pred High
Actual Low,"17,455",0
Actual High,0,"32,545"
,Pred Low,Pred High
Actual Low,"17,455",0
Actual High,0,"32,545"


 ### Train Decision Tree on Encrypted Data

 Build a depth-3 decision tree using Gini impurity from the same encrypted aggregate
 counts. Compare against sklearn's `DecisionTreeClassifier` (CART) as the real-world benchmark.

In [6]:
from blind_ml.demo_helpers import (
    run_encrypted_dt_fraud, fraud_dt_predict,
    train_plaintext_dt_fraud, fraud_plaintext_predict_proba,
    fraud_model_summary_table, build_raw_results_local,
    load_test_data, compute_fraud_metrics, fraud_confusion_matrix_html,
)

print("Training Decision Trees...")

# Reuse the encrypted aggregate counts NB already fetched from Blind Insight.
# DT's root split is chosen from these BI counts; deeper splits use local
# cross-tabs (see APPROACH.md — "root from BI marginals, deeper from local").
# LR consumes the same raw_results downstream — zero additional BI queries.
raw_results = bi["raw_results"]

# --- Encrypted DT (from BI aggregate counts, Gini criterion) ---
print("  Encrypted DT (Gini, depth 3)...", end=" ", flush=True)
enc_dt = run_encrypted_dt_fraud(
    raw_results=raw_results,
    feature_values=feature_values,
    df_local=df,
    n_high=bi["n_high"], n_low=bi["n_low"],
    max_depth=3, criterion="gini",
)
print(f"done ({enc_dt['train_time']:.2f}s)")

# --- Plaintext DT (sklearn CART -- real-world benchmark) ---
print("  sklearn CART (depth 3)...", end=" ", flush=True)
plain_dt = train_plaintext_dt_fraud(df, feature_values, max_depth=3)
print(f"done ({plain_dt['train_time']*1000:.0f}ms)")

# --- Evaluate on test set ---
df_test_dt, _ = load_test_data(TEST_SQLITE_DB)
COHORT_PRIOR = float(df_test_dt["is_high_risk"].mean())
y_true_dt = df_test_dt["is_high_risk"].values

enc_dt_scores = []
for _, row in df_test_dt.iterrows():
    _, risk = fraud_dt_predict(enc_dt, row.to_dict())
    enc_dt_scores.append(risk)
enc_dt_m = compute_fraud_metrics(y_true_dt, enc_dt_scores, cohort_prior=COHORT_PRIOR)

plain_dt_proba = fraud_plaintext_predict_proba(
    plain_dt["model"], plain_dt["col_names"], df_test_dt, feature_values)
plain_dt_m = compute_fraud_metrics(y_true_dt, plain_dt_proba, cohort_prior=COHORT_PRIOR)

print(
    f"\nEncrypted DT F1={enc_dt_m['f1']:.3f} ROC-AUC={enc_dt_m['roc_auc']:.3f} | "
    f"sklearn CART F1={plain_dt_m['f1']:.3f} ROC-AUC={plain_dt_m['roc_auc']:.3f}"
)
display(HTML(fraud_model_summary_table(
    "Decision Tree",
    enc_metrics=enc_dt_m,
    plain_metrics=plain_dt_m,
    enc_train_time=enc_dt["train_time"],
    plain_train_time=plain_dt["train_time"],
    enc_queries=enc_queries,
)))
display(HTML(fraud_confusion_matrix_html("Decision Tree", enc_dt_m, plain_dt_m)))

Training Decision Trees...
  Encrypted DT (Gini, depth 3)... done (1.29s)
  sklearn CART (depth 3)... done (1353ms)

Encrypted DT F1=1.000 ROC-AUC=1.000 | sklearn CART F1=1.000 ROC-AUC=1.000


,sklearn,Blind Insight,Delta
F1 @0.5 (demo prior),1.000,1.000,+0.000
F1@best (demo prior),1.000,1.000,+0.000
ROC-AUC,1.000,1.000,+0.000
PR-AUC,1.000,1.000,+0.000
F1@best @ 1.5% prod prior,1.000,1.000,+0.000
Sensitivity @0.5,100.0%,100.0%,+0.000
Specificity @0.5,100.0%,100.0%,+0.000
PPV (precision) @0.5,100.0%,100.0%,+0.000
Flagged High-Risk @0.5,65.1%,65.1%,+0.000
Train Time,1353ms,1.3s,+-0.1s


,Pred Low,Pred High
Actual Low,"17,455",0
Actual High,0,"32,545"
,Pred Low,Pred High
Actual Low,"17,455",0
Actual High,0,"32,545"


 ### Train Logistic Regression on Encrypted Data

 Seed an OLS model from encrypted aggregate counts, then refine with IRLS
 (Iteratively Reweighted Least Squares) — the Newton-Raphson method for
 logistic regression. Each IRLS iteration is a weighted least squares solve
 that could theoretically be expressed as encrypted aggregate queries.
 Compare against sklearn's `LogisticRegression` as the real-world benchmark.

In [7]:
from blind_ml.demo_helpers import (
    compute_fraud_pairwise_local, build_fraud_linear_model, fraud_lr_predict,
    refine_with_irls, train_plaintext_lr, fraud_model_summary_table,
    compute_fraud_metrics, build_plaintext_row, fraud_confusion_matrix_html,
)

print("Training Linear Regression models...")

# --- Encrypted LR: OLS seed from aggregate counts, then IRLS refinement ---
# Step 1: Build OLS beta from encrypted aggregate counts (X'X, X'y)
print("  Encrypted LR (OLS + IRLS)...", end=" ", flush=True)
lr_start = time.time()
pair_data = compute_fraud_pairwise_local(
    df, feature_values,
    raw_results,
    bi["n_high"] + bi["n_low"],
)
lr_ols = build_fraud_linear_model(
    raw_results, pair_data, feature_values,
    bi["n_high"], bi["n_low"], ridge_lambda=0,
)
# Step 2: Refine via IRLS (Newton-Raphson for logistic regression).
# Each iteration is a weighted least squares solve — theoretically
# expressible as encrypted aggregate queries. We run locally for speed
# since the local mirror matches BI 100%.
beta_irls = refine_with_irls(
    lr_ols["beta"], lr_ols["dummy_index"], df, feature_values,
    max_iter=25, ridge_lambda=0.01,
)
lr_model = {**lr_ols, "beta": beta_irls}
enc_lr_time = time.time() - lr_start
print(f"done ({enc_lr_time:.1f}s)")

# --- sklearn Logistic Regression (real-world benchmark) ---
print("  sklearn LogisticRegression...", end=" ", flush=True)
plain_lr_start = time.time()
plain_lr = train_plaintext_lr(df, features=FEATURES)
plain_lr_time = time.time() - plain_lr_start
print(f"done ({plain_lr_time*1000:.0f}ms)")

# --- Evaluate on test set ---
y_true_lr = df_test_dt["is_high_risk"].values

enc_lr_scores = []
for _, row in df_test_dt.iterrows():
    prob = fraud_lr_predict(
        lr_model["beta"], lr_model["dummy_index"], row.to_dict(), use_sigmoid=True
    )
    enc_lr_scores.append(prob)
enc_lr_m = compute_fraud_metrics(y_true_lr, enc_lr_scores, cohort_prior=COHORT_PRIOR)

plain_lr_scores = []
pos_idx = list(plain_lr["model"].classes_).index(1) if 1 in plain_lr["model"].classes_ else 0
for _, row in df_test_dt.iterrows():
    row_enc = build_plaintext_row(plain_lr["feature_columns"], row)
    plain_lr_scores.append(float(plain_lr["model"].predict_proba(row_enc)[0][pos_idx]))
plain_lr_m = compute_fraud_metrics(y_true_lr, plain_lr_scores, cohort_prior=COHORT_PRIOR)

print(
    f"\nEncrypted LR F1={enc_lr_m['f1']:.3f} ROC-AUC={enc_lr_m['roc_auc']:.3f} | "
    f"sklearn LR F1={plain_lr_m['f1']:.3f} ROC-AUC={plain_lr_m['roc_auc']:.3f}"
)
display(HTML(fraud_model_summary_table(
    "Logistic Regression",
    enc_metrics=enc_lr_m,
    plain_metrics=plain_lr_m,
    enc_train_time=enc_lr_time,
    plain_train_time=plain_lr_time,
    enc_queries=enc_queries,
)))
display(HTML(fraud_confusion_matrix_html("Logistic Regression", enc_lr_m, plain_lr_m)))

Training Linear Regression models...
  Encrypted LR (OLS + IRLS)... done (6.4s)
  sklearn LogisticRegression... done (2258ms)

Encrypted LR F1=0.000 ROC-AUC=0.500 | sklearn LR F1=1.000 ROC-AUC=1.000


,sklearn,Blind Insight,Delta
F1 @0.5 (demo prior),1.000,0.000,-1.000
F1@best (demo prior),1.000,0.789,-0.211
ROC-AUC,1.000,0.500,-0.500
PR-AUC,1.000,0.651,-0.349
F1@best @ 1.5% prod prior,1.000,0.789,-0.211
Sensitivity @0.5,100.0%,0.0%,-1.000
Specificity @0.5,100.0%,100.0%,+0.000
PPV (precision) @0.5,100.0%,0.0%,-1.000
Flagged High-Risk @0.5,65.1%,0.0%,-0.651
Train Time,2258ms,6.4s,+4.1s


,Pred Low,Pred High
Actual Low,"17,455",0
Actual High,"32,545",0
,Pred Low,Pred High
Actual Low,"17,455",0
Actual High,0,"32,545"


 ### Train Histogram Classifier on Encrypted Data

 Build a categorical histogram classifier from encrypted aggregate counts. Each
 feature value gets a smoothed risk bucket, then predictions average the matching
 buckets for a row. Compare against the same histogram algorithm trained on the
 plaintext local mirror.

In [8]:
from blind_ml.demo_helpers import (
    run_encrypted_histogram_fraud, train_plaintext_histogram_fraud,
    fraud_histogram_predict, fraud_model_summary_table,
    fraud_confusion_matrix_html, training_summary_table,
    load_test_data, discover_feature_values, compute_fraud_metrics,
)

print("Training Histogram Classifier...")

if "feature_values" not in globals():
    feature_values = discover_feature_values(df)
    feature_values["bank_ids"] = ["BANK001", "BANK002", "BANK003", "BANK014"]

n_high_hist = int((df["risk_level"] >= 50).sum())
n_low_hist = len(df) - n_high_hist

# --- Encrypted Histogram Classifier ---
print("  Encrypted Histogram...", end=" ", flush=True)
hist_enc = run_encrypted_histogram_fraud(
    client, ORG, DATASET, SCHEMA, feature_values,
    n_high=n_high_hist, n_low=n_low_hist,
)
print(f"done ({hist_enc['train_time']:.1f}s)")

# --- Plaintext Histogram Classifier ---
print("  Plaintext Histogram...", end=" ", flush=True)
hist_plain = train_plaintext_histogram_fraud(df, feature_values)
print(f"done ({hist_plain['train_time']*1000:.0f}ms)")

# --- Results ---
display(HTML(training_summary_table(
    hist_plain["n_high"], hist_plain["n_low"],
    hist_enc["n_high"], hist_enc["n_low"],
    hist_enc["enc_queries"], hist_enc["train_time"], hist_plain["train_time"],
)))
print(f"\n{hist_enc['enc_queries']} encrypted aggregate queries, P(high)={hist_enc['_model'].P_pos:.3f}")

# --- Evaluate on test set ---
if "df_test_dt" not in globals():
    df_test_dt, _ = load_test_data(TEST_SQLITE_DB)
if "COHORT_PRIOR" not in globals():
    COHORT_PRIOR = float(df_test_dt["is_high_risk"].mean())

y_true_hist = df_test_dt["is_high_risk"].values

hist_enc_scores = []
hist_plain_scores = []
for _, row in df_test_dt.iterrows():
    row_dict = row.to_dict()
    hist_enc_scores.append(fraud_histogram_predict(hist_enc, row_dict)[1])
    hist_plain_scores.append(fraud_histogram_predict(hist_plain, row_dict)[1])

hist_enc_m = compute_fraud_metrics(
    y_true_hist,
    hist_enc_scores,
    threshold=hist_enc["_model"].threshold,
    cohort_prior=COHORT_PRIOR,
)
hist_plain_m = compute_fraud_metrics(
    y_true_hist,
    hist_plain_scores,
    threshold=hist_plain["_model"].threshold,
    cohort_prior=COHORT_PRIOR,
)

print(
    f"\nEncrypted Histogram F1={hist_enc_m['f1']:.3f} ROC-AUC={hist_enc_m['roc_auc']:.3f} | "
    f"Plaintext F1={hist_plain_m['f1']:.3f} ROC-AUC={hist_plain_m['roc_auc']:.3f}"
)
display(HTML(fraud_model_summary_table(
    "Histogram Classifier",
    enc_metrics=hist_enc_m,
    plain_metrics=hist_plain_m,
    enc_train_time=hist_enc["train_time"],
    plain_train_time=hist_plain["train_time"],
    enc_queries=hist_enc["enc_queries"],
    plain_label="Plaintext Histogram",
)))
display(HTML(fraud_confusion_matrix_html("Histogram Classifier", hist_enc_m, hist_plain_m)))


Training Histogram Classifier...
  Encrypted Histogram... done (17.9s)
  Plaintext Histogram... done (1779ms)


,Plaintext,Blind Insight,Overhead
High Risk,"326,472","326,472",-
Low Risk,"173,528","173,528",-
Total,"500,000","500,000",-
Queries,0,90,-
Train Time,1.778618s,17.9s,+16.2s
Data Decrypted,YES,NEVER,-



90 encrypted aggregate queries, P(high)=0.653

Encrypted Histogram F1=1.000 ROC-AUC=1.000 | Plaintext F1=1.000 ROC-AUC=1.000


,Plaintext Histogram,Blind Insight,Delta
F1 @0.5 (demo prior),1.000,1.000,+0.000
F1@best (demo prior),1.000,1.000,+0.000
ROC-AUC,1.000,1.000,+0.000
PR-AUC,1.000,1.000,+0.000
F1@best @ 1.5% prod prior,1.000,1.000,+0.000
Sensitivity @0.5,100.0%,100.0%,+0.000
Specificity @0.5,100.0%,100.0%,+0.000
PPV (precision) @0.5,100.0%,100.0%,+0.000
Flagged High-Risk @0.5,65.1%,65.1%,+0.000
Train Time,1779ms,17.9s,+16.2s


,Pred Low,Pred High
Actual Low,"17,455",0
Actual High,0,"32,545"
,Pred Low,Pred High
Actual Low,"17,455",0
Actual High,0,"32,545"


 ### Six-Model Comparison: Encrypted vs sklearn Benchmarks

 Side-by-side metrics on the demo test set: **F1 @0.5** (balanced ~65% prior), **ROC-AUC** (prior-invariant), **PR-AUC**, and **F1@best** after recalibrating scores to a **1.5% production fraud prior**.

In [9]:
from blind_ml.demo_helpers import (
    fraud_three_model_table, fraud_confusion_matrix_html,
    compute_fraud_metrics, naive_bayes_predict_proba, FRAUD_PRODUCTION_PRIOR,
)

# NB metrics (recompute from test set for consistency)
nb_enc_scores = []
nb_plain_scores = []
for _, row in df_test_dt.iterrows():
    r = row.to_dict()
    nb_enc_scores.append(naive_bayes_predict_proba(P_high, P_low, P_tables, r))
    nb_plain_scores.append(naive_bayes_predict_proba(P_high_plain, P_low_plain, P_tables_plain, r))
nb_m = compute_fraud_metrics(y_true_dt, nb_enc_scores, cohort_prior=COHORT_PRIOR)
nb_plain_m = compute_fraud_metrics(y_true_dt, nb_plain_scores, cohort_prior=COHORT_PRIOR)

print(f"Six-Model Comparison ({len(df_test_dt):,} test records, cohort prior={COHORT_PRIOR:.1%})")
print(f"Production prior for F1@best column: {FRAUD_PRODUCTION_PRIOR:.1%}")
for label, enc_m, plain_m in [
    ("NB", nb_m, nb_plain_m),
    ("BN", bn_enc_m, bn_plain_m),
    ("GNB", gnb_enc_m, gnb_plain_m),
    ("DT", enc_dt_m, plain_dt_m),
    ("LR", enc_lr_m, plain_lr_m),
    ("HIST", hist_enc_m, hist_plain_m),
]:
    print(
        f"  {label:4} enc F1={enc_m['f1']:.3f} ROC-AUC={enc_m['roc_auc']:.3f} "
        f"F1@prod={enc_m['f1_prod_best']:.3f} | plain F1={plain_m['f1']:.3f}"
    )

model_rows = [
    {"name": "Naive Bayes", "enc_metrics": nb_m},
    {"name": "Bayesian Net", "enc_metrics": bn_enc_m},
    {"name": "Gaussian NB", "enc_metrics": gnb_enc_m},
    {"name": "Decision Tree", "enc_metrics": enc_dt_m},
    {"name": "Logistic Reg", "enc_metrics": enc_lr_m},
    {"name": "Histogram", "enc_metrics": hist_enc_m},
]
display(HTML(fraud_three_model_table(model_rows)))

print("\nConfusion Matrices (test set, demo threshold):")
display(HTML(
    fraud_confusion_matrix_html("Naive Bayes", nb_m, nb_plain_m)
    + fraud_confusion_matrix_html("Bayesian Network", bn_enc_m, bn_plain_m)
    + fraud_confusion_matrix_html("Gaussian Naive Bayes", gnb_enc_m, gnb_plain_m)
    + fraud_confusion_matrix_html("Decision Tree", enc_dt_m, plain_dt_m)
    + fraud_confusion_matrix_html("Logistic Regression", enc_lr_m, plain_lr_m)
    + fraud_confusion_matrix_html("Histogram Classifier", hist_enc_m, hist_plain_m)
))
print(f"\n→ All 6 models trained from encrypted aggregate queries, zero records decrypted.")


Six-Model Comparison (50,000 test records, cohort prior=65.1%)
Production prior for F1@best column: 1.5%
  NB   enc F1=1.000 ROC-AUC=1.000 F1@prod=1.000 | plain F1=1.000
  BN   enc F1=1.000 ROC-AUC=1.000 F1@prod=1.000 | plain F1=1.000
  GNB  enc F1=0.789 ROC-AUC=0.505 F1@prod=0.789 | plain F1=0.789
  DT   enc F1=1.000 ROC-AUC=1.000 F1@prod=1.000 | plain F1=1.000
  LR   enc F1=0.000 ROC-AUC=0.500 F1@prod=0.789 | plain F1=1.000
  HIST enc F1=1.000 ROC-AUC=1.000 F1@prod=1.000 | plain F1=1.000


Metric,Naive Bayes,Bayesian Net,Gaussian NB,Decision Tree,Logistic Reg,Histogram
F1 @0.5,1.000,1.000,0.789,1.000,0.000,1.000
F1@best,1.000,1.000,0.789,1.000,0.789,1.000
ROC-AUC,1.000,1.000,0.505,1.000,0.500,1.000
PR-AUC,1.000,1.000,0.654,1.000,0.651,1.000
F1@best @ 1.5% prod,1.000,1.000,0.789,1.000,0.789,1.000
Accuracy @0.5,100.0%,100.0%,65.1%,100.0%,34.9%,100.0%



Confusion Matrices (test set, demo threshold):


,Pred Low,Pred High
Actual Low,"17,455",0
Actual High,0,"32,545"
,Pred Low,Pred High
Actual Low,"17,455",0
Actual High,0,"32,545"
,Pred Low,Pred High
Actual Low,"17,455",0
Actual High,0,"32,545"
,Pred Low,Pred High
Actual Low,"17,455",0



→ All 6 models trained from encrypted aggregate queries, zero records decrypted.


 ### Real-Time Decision Making on Encrypted Application Data

 Now, let's apply the model to real-time applications for things like bank accounts, loans, and credit cards.

In [10]:
from blind_ml.demo_helpers import run_realtime_demo

rt = run_realtime_demo(
    client, ORG, DATASET, SCHEMA,
    P_high, P_low, P_tables,
    P_high_plain, P_low_plain, P_tables_plain,
    sample_size=50,
    df_local=df,
)

n = rt["rt_count"]
print(
    f"Query: {rt['enc_query_time'] * 1000:.0f}ms for {n} records | "
    f"Predict: BI={rt['bi_avg_ms']:.2f}ms, Plain={rt['plain_avg_ms']:.2f}ms per record"
)

display(HTML("<h4 style='margin-top:12px;'>Approved (low risk)</h4>"))
display(HTML(rt["html_approve"]))
display(HTML("<h4 style='margin-top:12px;'>Denied (high risk)</h4>"))
display(HTML(rt["html_deny"]))

Query: 1075ms for 50 records | Predict: BI=0.00ms, Plain=0.01ms per record


BI (ms),fraud_type,account_jurisdiction,is_active,month,reporting_bank_id,year,risk_level,Plain NB,BI NB,Match
21,e6721616bdb56007f69dcab931970184956,c39466835cbde937e509917b9bdcce3c517,ec5b6a2d43f6199e64966cc954f3d6dd42c,dfd198f1c2d2872499f19a30d9db68ef962,f0c98d8607e3069e39c629962406200e52e,71797648819a6b523bbbc23dde2dbc757f4,099e2db95e6ac5ee57c390cb8019e9e3b29,✅,✅,✓
21,6aca8751646cdc677e3b5e96ff18d2f04f8,b8848a00ac12f24651701b5cce2403aa590,79b7aab34a42f1639775242269c8978da74,6b9c924d306b43b23ffeffe4a5638779f71,68c84ecaeaca07c8d3174667df7ea3a38bb,fd7bb2606f9600b60b8decbf0cb7408f4dd,4735f206d3750ed4ea35688ad8664307d28,✅,✅,✓
21,67b09d498ac6591eb3fae074eec8bad74c1,0f179a85155e8bed7cae515c1e42f00ad3c,20f4b7477c7c862e34cac2a8cb1f7f4a78e,ac639e60347742cced57727fd9789f66e7a,0c998d554fc9f8ea5e489951719e3c6c6ad,7c697e055332e5d661d60f1153880814f33,5f70f88242e45a0903a948858cb25da33e4,✅,✅,✓
21,e3cb68a5e2c1cb905e282c58946ae29ffed,2079b8d72441a05ae0e565d722be7ff0b18,4d8c416b761e74723dbb2fcfbcee373b5bc,bf42d57a3c92507319f7cc1732389cfb11d,a65eb933340d2bf3d73ee870da0c24de4b8,1a1dc24a8ab817cbd2f266dcb5dda750546,8b8ffc7ff76bbe4a2afa7aa95abb19acb3c,✅,✅,✓
21,a3f7967fee76b9af07a76232afb2b9f65a9,faf680810bf2ae9d89671fae767f4ce815f,cf67f2441368685dcbcbd5645605c686a62,6110aafdca7dc340fa73d0a0f3d538ee1df,7ddb7efc5b00ed56268a708c9795a2fba53,c49a21f2cd4a04b50d1766eb8ee571f0801,5cd8fafd537b96d19dc261cfe16c26e66f5,✅,✅,✓


BI (ms),fraud_type,account_jurisdiction,is_active,month,reporting_bank_id,year,risk_level,Plain NB,BI NB,Match
21,3253a21d39063282c4cb7e8b54f24f0e41c,30fb178c0aa8260973b90465008fef6d3d3,94987a257e705fa5e6556c793b4e485dd08,d7e0a1536b958f8411674f96f2c8fc5f35d,940097d999744955f878461f5d1c7fdea8f,03ee88f5071b499dd4deb80203994408453,76e1ac0e324e9b56942c4c70bb94909e37a,❌,❌,✓
21,e6af46f6063c170ed8420b9e3e808397ee2,67a0be500ab69545b1ce3867fc79521ea64,c102c1825be989d8de1c3406c193730572d,32f76f60f4045a415c1034145984d3b7d94,2322b18ce29c7f5f70d29e26656baebc13d,081ebbb8cfacdc7b3a2d0a54f49f6fc5b05,76a49188435a1c724de2353d1ceb3557cd7,❌,❌,✓
21,2d6b031eb8b7c94b2f34b8f45da4a0319c2,b893246dcae4e7f0e999adf5a9d2bb3f343,a4ff42e5577a263d7dd3d020c5dd375a1bf,e52187f5552d9b40cab99f688545b3fce7f,adf80d721758c479ae89fbeab1208f95d03,5b3fca2c52abc6dd469ada51797978a223c,6dec9fdf492d1b3e068fe4e47bf4596c3d9,❌,❌,✓
21,59c1096f8c49bba604dff58ffbfce654e37,2497b76e1e9eac2f2c1b9f1e7270c0e3154,f4759074dba9aaf85df58c6c5f9cf0fdc4b,bf2ca05721ed179f146b3d0f0667f0aea1c,87ca4f0027538585d9af223fc11e1e07f11,34a9d1d8ab230daac896a9c0664d945c94b,be5fef274a6885799b6b19e1d5cc64f59c6,❌,❌,✓
21,55fedeb8cd26804db1c6230d8d2801c01bc,684efbfa1a964be8306d9dfbd92a5d28e96,fffe42fdda921dbccc54d2ca48a8fbf8176,bda8b4f241e7520777b95bfb630192ad4dd,59c1d21f32d0f80a8834c7b632b8f913359,5260260562a06cd6341762e56a85a72dcd1,c5355095a3016b17ceced91bfbdfef39c23,❌,❌,✓


 ### Validate Encrypted Model Against Plaintext on 50K Test Records

In [11]:
from blind_ml.demo_helpers import run_test_validation

# Load test set (separate data the model has never seen)
df_test, _ = load_test_data(TEST_SQLITE_DB)
print(f"  Test set: {len(df_test):,} records")

# Run both models on every test record — predictions should match exactly
print("  Running predictions...", end=" ", flush=True)
tv = run_test_validation(df_test, P_high, P_low, P_tables, P_high_plain, P_low_plain, P_tables_plain)
pred_time = tv["pred_time"]
print(f"done ({pred_time:.1f}s) | Agreement: {tv['agreement']*100:.1f}% | Accuracy: {tv['acc_enc']*100:.1f}%")

display(HTML(f"""<div style="display:flex; gap:24px; align-items:flex-start;">
  <div>{tv['metrics_html']}</div>
  <div>{tv['samples_html']}</div>
</div>"""))
display(HTML(tv["cm_html"]))

  Test set: 50,000 records
  Running predictions... done (1.3s) | Agreement: 100.0% | Accuracy: 100.0%


,Plaintext NB,Blind Insight NB,Match
Records,"50,000","50,000",OK
High Risk,"32,545","32,545",OK
Low Risk,"17,455","17,455",OK
BI <-> Plain Agreement,-,100.0%,OK
Model Accuracy,100.0%,100.0%,OK
Prediction Loop Time,1.32s,1.32s,-
Fraud Type,Jurisdiction,Risk,Decision
identity_theft,AU,93,❌
unusual_activity,US,34,✅
mule_account,AU,99,❌


,Pred Low,Pred High
Actual Low,"17,455",0
Actual High,0,"32,545"
,Pred Low,Pred High
Actual Low,"17,455",0
Actual High,0,"32,545"


 ### Scaling Comparison + Interactive Calculator

In [12]:
from blind_ml.demo_helpers import scaling_calculator_html

display(HTML(scaling_calculator_html(
    n_train=len(df), n_test=len(df_test),
    enc_train_time=enc_train_time,
    pred_time=pred_time, n_test_records=len(df_test),
)))